# 05 — Elo rating

Adds an Elo rating as a feature: one number per fighter, corrected after every fight, so it
is the only feature that knows how good a fighter's opponents were
(`src/ufc_predictor/features/elo.py`).

Since Day 12 a win is weighted by **how it came**: a finish counts 1, a unanimous decision
less, a split or majority decision less again, and the loser gets the mirror value. The
source's `method` column exists only for this; it is never a model input.

What shapes the notebook:

- **Elo is judged by the model it serves.** K and the two win weights are searched together,
  scored by cross-validation with *the model's own current settings*. On Day 12 a K judged by
  the model tuned without Elo (a differently shaped model) lost to the previous K in 8 of 8
  seeds once checked with the final settings.
- **Ties go to the smallest change.** Settings within 0.0005 of the best are a tie; among them
  the one closest to the previous Elo is chosen.
- **A new Elo must earn its place.** It gets its own tuned settings and must beat the previous
  Elo in at least 7 of 8 seeds without being worse on men's validation.

Order: tune without Elo → Elo alone (reference) → search K and weights → check the choice →
rebuild the table → settings with Elo → with vs without Elo. The test years stay closed.
Runtime: about an hour and a quarter.

In [1]:
import numpy as np
import optuna
import pandas as pd

optuna.logging.set_verbosity(optuna.logging.WARNING)   # one line per trial would bury the results

from ufc_predictor.data.io import load_fighters, load_processed, save_processed
from ufc_predictor.features import manifest
from ufc_predictor.features.elo import (ELO_COLUMNS, elo_forecast_log_loss, load_elo_params,
                                        load_elo_weights, save_elo_params)
from ufc_predictor.features.pipeline import build_featured_table
from ufc_predictor.features.shrinkage import load_alphas
from ufc_predictor.modeling.evaluate import compare, group_importance, is_womens_fight, score, scorecard
from ufc_predictor.modeling.model import fit_model, predict_proba
from ufc_predictor.modeling.split import chronological_split
from ufc_predictor.modeling.tune import (choose_elo_settings, cv_log_loss, load_params, pick_elo_setting,
                                         save_params, tune, tuning_table)

fights = load_processed("fights")
fighters = load_fighters()
alphas = load_alphas()

# what the model uses before this run: the judge for the Elo search, and the incumbent
previous = {"k": load_elo_params(), "weights": load_elo_weights()}
params_previous = load_params()
print(f"previous Elo: K {previous['k']}, weights {previous['weights'] or 'none (classic)'}")

without_elo = build_featured_table(fights, fighters, alphas, elo_k=None, elo_weights=None)
train_without, validation_without, _ = chronological_split(without_elo)
print(f"{without_elo.shape[0]:,} fights x {without_elo.shape[1]} columns | training fights: {len(train_without):,}")

SEEDS = range(8)
mens = ~is_womens_fight(validation_without, history=without_elo).to_numpy()
y = np.asarray(manifest.make_xy(validation_without)[1], dtype=bool)

def seed_averaged(train_table, valid_table, params):
    """Validation predictions averaged over SEEDS, plus the seed-0 model."""
    X_tr, y_tr = manifest.make_xy(train_table)
    X_va, _ = manifest.make_xy(valid_table)
    models = [fit_model(X_tr, y_tr, params=params | {"random_state": s}) for s in SEEDS]
    return np.mean([predict_proba(m, X_va) for m in models], axis=0), models[0], X_va

previous Elo: K 251.2, weights none (classic)


8,937 fights x 371 columns | training fights: 4,309


## 1. Tune the model without Elo

Gives the "without Elo" settings that the K search and the final comparison
need, and restores `model_params.json` honestly. Same search as before (60
Optuna trials, seeded), but each trial is now averaged over 3 model seeds so
the search can't reward a lucky draw.

In [2]:
study_without = tune(train_without, n_trials=60)
save_params(study_without)
params_without = study_without.best_params
print(f"best CV log loss, averaged over 3 seeds: {study_without.best_value:.5f}")
tuning_table(study_without, top=5)

best CV log loss, averaged over 3 seeds: 0.66097


,trial,log_loss,learning_rate,max_iter,max_leaf_nodes,min_samples_leaf,l2_regularization,max_features
0,46,0.66097,0.01619,550,15,8,2.29850,0.68860
1,53,0.66119,0.01325,700,18,6,3.17521,0.63208
2,55,0.66138,0.01312,700,20,6,2.29220,0.72258
3,59,0.66145,0.01194,800,18,5,1.91275,0.54129
4,48,0.66152,0.01158,650,19,5,2.05611,0.62829


## 2. Elo on its own, for reference

Formula 1 used directly as a forecast, scored on the training years (2014–2022). Every forecast
uses only earlier fights, so this never touches later years. This is the
table from the research page, and it is **not** how K is chosen: on its own,
Elo is best around K ≈ 63, while the model wants something faster.

In [3]:
pd.DataFrame([
    {"K": k, "elo_alone_log_loss": round(elo_forecast_log_loss(fights, k), 5)}
    for k in (16, 31.62, 63.1, 100, 158.5, 199.5, 316.2)
])

,K,elo_alone_log_loss
0,16.00,0.68770
1,31.62,0.68445
2,63.10,0.68151
3,100.00,0.68240
4,158.50,0.69156
5,199.50,0.70302
6,316.20,0.75614


## 3. Search K and the win weights, judged by the model's own settings

Every combination of 9 values of K (40 to 400) and 20 pairs of weights (unanimous 1.0 → 0.6,
narrow 1.0 → 0.5, narrow never above unanimous): 180 settings, each scored by
expanding-window CV over 5 seeds with the settings the model currently uses. The previous
Elo is one of the settings, so if nothing beats it, it stays.

In [4]:
curve = choose_elo_settings(without_elo, params_previous)
choice = pick_elo_setting(curve, previous)
current = curve[(curve.K == previous["k"])
                & (curve.unanimous == (previous["weights"] or {}).get("unanimous", 1.0))
                & (curve.narrow == (previous["weights"] or {}).get("narrow", 1.0))]
print(f"previous Elo on this grid: {current.cv_log_loss.iloc[0]:.5f}   best: {curve.cv_log_loss.min():.5f}")
print(f"{choice['near_best']} settings within 0.0005 of the best; closest to the previous Elo: "
      f"K {choice['k']}, weights {choice['weights']}, CV {choice['cv_log_loss']:.5f}")
print("best per K:")
print(curve.loc[curve.groupby("K").cv_log_loss.idxmin()].sort_values("K").round(5).to_string(index=False))
curve.head(10).round(5)

previous Elo on this grid: 0.65753   best: 0.65512
6 settings within 0.0005 of the best; closest to the previous Elo: K 158.5, weights {'unanimous': 0.7, 'narrow': 0.55}, CV 0.65561
best per K:
     K  unanimous  narrow  cv_log_loss
 39.81        1.0    0.55      0.65684
 63.10        0.9    0.50      0.65652
100.00        0.8    0.50      0.65558
125.90        0.7    0.50      0.65546
158.50        0.7    0.50      0.65512
199.50        0.7    0.55      0.65605
251.20        0.7    0.65      0.65635
316.20        0.7    0.65      0.65653
398.10        0.7    0.65      0.65730


,K,unanimous,narrow,cv_log_loss
0,158.5,0.7,0.50,0.65512
1,125.9,0.7,0.50,0.65546
2,125.9,0.8,0.50,0.65554
3,100.0,0.8,0.50,0.65558
4,158.5,0.7,0.55,0.65561
5,125.9,0.9,0.55,0.65562
6,100.0,0.9,0.50,0.65570
7,158.5,0.8,0.50,0.65573
8,100.0,0.8,0.65,0.65575
9,100.0,0.7,0.50,0.65591


## 4. Check the choice against the previous Elo

The chosen Elo gets its own tuned settings (60 trials, 3 seeds) and is compared with the
previous Elo and *its* settings, over 8 seeds. The rule, set before running: adopt the new
Elo only if CV is better in at least 7 of 8 seeds and men's validation log loss is not worse.

In [5]:
weights_or_classic = lambda w: w or {"unanimous": 1.0, "narrow": 1.0}
table_previous = build_featured_table(fights, fighters, alphas, elo_k=previous["k"], elo_weights=previous["weights"])
train_previous, validation_previous, _ = chronological_split(table_previous)
adopt, study_new = False, None
if (choice["k"], choice["weights"]) != (previous["k"], weights_or_classic(previous["weights"])):
    table_new = build_featured_table(fights, fighters, alphas, elo_k=choice["k"], elo_weights=choice["weights"])
    train_new, validation_new, _ = chronological_split(table_new)
    study_new = tune(train_new, n_trials=60)
    change = (np.array(cv_log_loss(train_new, study_new.best_params, seeds=SEEDS)["log_loss_by_seed"])
              - np.array(cv_log_loss(train_previous, params_previous, seeds=SEEDS)["log_loss_by_seed"]))
    p_new, _, _ = seed_averaged(train_new, validation_new, study_new.best_params)
    p_prev, _, _ = seed_averaged(train_previous, validation_previous, params_previous)
    v_new, v_prev = score(y[mens], p_new[mens])["log_loss"], score(y[mens], p_prev[mens])["log_loss"]
    adopt = (change < 0).sum() >= 7 and v_new <= v_prev
    print(f"new Elo better in {(change < 0).sum()} of 8 seeds (mean change {change.mean():+.5f}); "
          f"men's validation log loss {v_new:.4f} vs {v_prev:.4f}")
best = choice if adopt else previous
note = (f"searched K and win weights with the model's own settings; chosen K {choice['k']}, weights "
        f"{choice['weights']}; {'adopted' if adopt else 'not adopted, previous kept'} by the rule (05_elo.ipynb, section 4)")
save_elo_params(best["k"], curve.to_dict("records"), seeds=range(5), weights=best["weights"], note=note)
print(note)

new Elo better in 8 of 8 seeds (mean change -0.00283); men's validation log loss 0.6402 vs 0.6455
searched K and win weights with the model's own settings; chosen K 158.5, weights {'unanimous': 0.7, 'narrow': 0.55}; adopted by the rule (05_elo.ipynb, section 4)


## 5. Rebuild the featured table with Elo

The same shared chain as notebook 03, now with the Elo from section 4. Spot-check: UFC 69,
where Matt Serra upset Georges St-Pierre.

In [6]:
featured = build_featured_table(fights, fighters, alphas, elo_k=best["k"], elo_weights=best["weights"])
save_processed(featured, "fights_featured")

assert manifest.unclassified_columns(featured) == []
print(f"{featured.shape[1]} columns, {len(manifest.feature_columns(featured))} features")
featured.loc[
    featured["event_date"].eq("2007-04-07") & featured["fighter_a_name"].eq("Matt Serra"),
    ["fighter_a_name", "fighter_b_name", *ELO_COLUMNS, "diff_prior_elo"],
].round(1)

374 columns, 40 features


,fighter_a_name,fighter_b_name,fighter_a_prior_elo,fighter_b_prior_elo,diff_prior_elo
713,Matt Serra,Georges St-Pierre,1536.4,1882.1,-345.7


## 6. Settings with Elo

If section 4 adopted the new Elo, its tuning there was on exactly this table and is reused;
otherwise the settings are re-tuned the same way. This saves `model_params.json`.

In [7]:
train, validation, _ = chronological_split(featured)      # the test years stay closed
study_with = study_new if adopt else tune(train, n_trials=60)
save_params(study_with)
params_with = study_with.best_params
print(f"best CV log loss, averaged over 3 seeds: {study_with.best_value:.5f}")
tuning_table(study_with, top=5)

best CV log loss, averaged over 3 seeds: 0.65448


,trial,log_loss,learning_rate,max_iter,max_leaf_nodes,min_samples_leaf,l2_regularization,max_features
0,59,0.65448,0.01127,950,5,28,0.00342,0.54579
1,58,0.65468,0.01335,900,6,22,0.00665,0.52954
2,43,0.65472,0.01541,950,5,26,0.00226,0.35709
3,27,0.65472,0.02673,900,4,29,0.00248,0.47930
4,52,0.65473,0.01402,900,5,28,0.00352,0.41108


## 7. The decision

Each version with its own tuned settings. First cross-validation over 8 seeds,
compared seed by seed. Then the one look at validation, with predictions
averaged over the same 8 seeds. The rule set before running: keep Elo if CV
improves in at least 7 of 8 seeds and validation doesn't contradict it.

In [8]:
cv_without = cv_log_loss(train_without, params_without, seeds=SEEDS)
cv_with = cv_log_loss(train, params_with, seeds=SEEDS)
change = np.array(cv_with["log_loss_by_seed"]) - np.array(cv_without["log_loss_by_seed"])
print(f"CV log loss: without Elo {cv_without['log_loss']:.5f} -> with Elo {cv_with['log_loss']:.5f}")
print(f"change {change.mean():+.5f}, better in {(change < 0).sum()} of 8 seeds")

CV log loss: without Elo 0.66122 -> with Elo 0.65477
change -0.00645, better in 8 of 8 seeds


In [9]:
assert (validation_without["fights_url"].to_numpy() == validation["fights_url"].to_numpy()).all()

p_without, _, _ = seed_averaged(train_without, validation_without, params_without)
p_with, model_with, X_val = seed_averaged(train, validation, params_with)
_, y_val = manifest.make_xy(validation)

result = compare(y[mens], p_with[mens], p_without[mens])
print(f"men's fights, with vs without Elo: fixes {result['fixes']}, breaks {result['breaks']}, "
      f"p = {result['p_value']:.3f}")
scorecard({"without Elo": p_without, "with Elo": p_with}, y_val, mens)

men's fights, with vs without Elo: fixes 105, breaks 61, p = 0.001


,men_accuracy,men_log_loss,men_auc,all_accuracy,all_log_loss,all_auc
model,,,,,,
without Elo,0.5938,0.6551,0.6507,0.5939,0.6591,0.6430
with Elo,0.6464,0.6402,0.6892,0.6372,0.6450,0.6778


In [10]:
# which feature families the tuned model with Elo relies on (men's validation fights)
groups = manifest.feature_groups(train)
group_importance(model_with, X_val[mens], y_val[mens], groups, repeats=20)

,group,columns,log_loss_rise,spread
0,strength,1,0.028254,0.005126
1,physical,5,0.027398,0.003412
2,defence,11,0.013315,0.004902
3,striking,13,0.006750,0.003137
4,grappling,4,0.005536,0.002660
5,cold_start,2,0.000501,0.000779
6,record,3,0.000102,0.000632
7,damage,1,0.000068,0.000243
